# 从平方增量到随机积分
先修：正态增量、条件期望、正文的二次变差证明。依次观察同一路径网格、跨路径误差、Itô 平方公式和 GBM。随机种子用于复现，可自由修改；不要求某条路径输出固定的小数。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(2026)
T, steps = 1., 4096
dw = rng.normal(0, np.sqrt(T/steps), steps)
blocks = [64, 16, 4, 1]
qv = []
for block in blocks:
    increments = dw.reshape(-1, block).sum(axis=1)
    qv.append(increments @ increments)
print('步数与二次变差：', list(zip([steps//b for b in blocks], qv)))
fig, ax = plt.subplots(); ax.plot([steps//b for b in blocks], qv, 'o-')
ax.axhline(T, color='gray', linestyle='--'); ax.set_xscale('log')
ax.set(xlabel='steps', ylabel='quadratic variation'); plt.show()

同一路径的误差不必单调。定理描述概率收敛；均匀网格有 $\operatorname{Var}(Q)=2T^2/m$。下格重复模拟比较这个分布结论。

In [ ]:
for m in [32, 128, 512]:
    increments = rng.normal(0, np.sqrt(T/m), (1500, m))
    values = np.sum(increments**2, axis=1)
    print(m, '均值', values.mean(), '经验方差', values.var(), '理论', 2*T*T/m)
left = np.r_[0., np.cumsum(dw)[:-1]]
ito_sum = left @ dw
right_sum = (left+dw) @ dw
terminal = dw.sum()
print('左端和 / 极限公式：', ito_sum, .5*(terminal**2-T))
print('有限网格恒等式残差：', terminal**2-2*ito_sum-dw@dw)
print('右端和减左端和 / 二次变差：', right_sum-ito_sum, dw@dw)

左端和与极限公式的差恰为 $(T-Q)/2$；有限网格恒等式却每次都成立。右端取值使用当前区间末端信息，两种和之差趋于 $T$，说明取值时点会改变积分。

In [ ]:
mu, sigma, S0, r = .08, .2, 100., .02
wT = rng.normal(0, np.sqrt(T), 100000)
ST = S0*np.exp((mu-sigma*sigma/2)*T+sigma*wT)
print('价格均值：', ST.mean(), '理论：', S0*np.exp(mu*T))
print('对数收益均值：', np.log(ST/S0).mean(), '理论：', (mu-sigma*sigma/2)*T)
theta = (mu-r)/sigma
density = np.exp(-theta*wT-.5*theta*theta*T)
discounted = np.exp(-r*T)*ST
print('密度均值（理论 1）：', density.mean())
print('重加权贴现股票均值（理论 S0）：', np.mean(density*discounted))
print('漂移代换：', mu-sigma*theta)

这里用原测度下样本及密度权重估计新测度期望。常数参数的鞅性由正文的正态指数矩证明，模拟均值接近 1 只是观察。

## 练习
1. 将左端值换为下一端值，为何不能继续解释成相同的 Itô 积分？
2. 删除 GBM 中的 $-\sigma^2/2$ 后，价格理论均值变成什么？
3. 常数风险价格很大是否违反 Novikov？

反馈：第一题差为二次变差；第二题变成 $S_0e^{(\mu+\sigma^2/2)T}$；第三题不违反，有限常数的指数仍有限，但模拟权重可能高度不均，有限样本估计会不稳定。